## In this case i will use Game on Steam  https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam Dataset

#### Path to files

In [ ]:
import os
import pandas as pd

cwd = os.getcwd()

path_to_dir = os.path.join(cwd, 'data')
path_to_metadata = os.path.join(path_to_dir, 'games_metadata.json')
path_to_games = os.path.join(path_to_dir, 'games.csv')

#### Merging two df`s

In [2]:
games_info = pd.read_csv(path_to_games, index_col='app_id')
metadata_info = pd.read_json(path_to_metadata, lines=True).set_index('app_id')

df_ready = games_info.join(metadata_info, how='inner')

In [3]:
df_ready.head()

,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck,description,tags
app_id,,,,,,,,,,,,,,
13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True,,[Action]
113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


In [4]:
df_ready.shape

(50872, 14)

#### Converting ratings to numeric data

In [5]:
df_ready['rating'].unique()

<StringArray>
[          'Very Positive',                'Positive',
                   'Mixed',         'Mostly Positive',
 'Overwhelmingly Positive',                'Negative',
         'Mostly Negative', 'Overwhelmingly Negative',
           'Very Negative']
Length: 9, dtype: str

In [6]:
rating_map = {
    'Overwhelmingly Positive': 10,
    'Very Positive': 9,
    'Positive': 8,
    'Mostly Positive': 6,
    'Mixed': 5,
    'Mostly Negative': 3,
    'Negative': 2,
    'Very Negative': 1,
    'Overwhelmingly Negative': 0
}

df_ready['rating'] = df_ready['rating'].map(rating_map)

In [7]:
df_ready.head()

,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck,description,tags
app_id,,,,,,,,,,,,,,
13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,9,84,2199,9.99,9.99,0.0,True,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
22364,BRINK: Agents of Change,2011-08-03,True,False,False,8,85,21,2.99,2.99,0.0,True,,[Action]
113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,9,92,3722,14.99,14.99,0.0,True,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
226560,Escape Dead Island,2014-11-18,True,False,False,5,61,873,14.99,14.99,0.0,True,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,9,88,8784,11.99,11.99,0.0,True,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


# Simple recommender implementation

#### Calculate mean of raiting

In [8]:
C = df_ready['rating'].mean()
print(C)

7.01354379619437


#### Calculate the minimum number of reviews required to be in the result

In [9]:
m = df_ready['user_reviews'].quantile(0.75)
print(m)

206.0


#### Filtering

In [10]:
q_games = df_ready.copy().loc[df_ready['user_reviews'] >= m]
q_games.shape


(12752, 14)

#### Function that computes the weighted rating of each game


In [11]:
def weighted_rating(x, m=m, C=C):
    v = x['user_reviews']
    R = x['positive_ratio'] / 10
    return (v / (v + m) * R) + (m / (m + v) * C) # formula IMDB

In [12]:
# Define a new feature 'score' and calculate its value with `weighted_rating()`
q_games['score'] = q_games.apply(weighted_rating, axis=1)

#Sort games based on score calculated above
q_games = q_games.sort_values('score', ascending=False)

#Print the top 15 games
q_games[['title', 'user_reviews', 'positive_ratio', 'score']].head(15)


,title,user_reviews,positive_ratio,score
app_id,,,,
431730,Aseprite,11608,99,9.849669
1144400,Senren＊Banka,11410,99,9.848811
1055540,A Short Hike,10534,99,9.844636
431960,Wallpaper Engine,637341,98,9.799100
413150,Stardew Valley,505882,98,9.798866
620,Portal 2,293053,98,9.798043
1145360,Hades,214267,98,9.797324
1794680,Vampire Survivors,197109,98,9.797091
1118200,People Playground,195164,98,9.797062


# Content-Based Recommender

In [13]:
# tags
df_ready['tags'].head()


app_id
13500     [Action, Adventure, Parkour, Third Person, Gre...
22364                                              [Action]
113020    [Co-op, Stealth, Indie, Heist, Local Co-Op, St...
226560    [Zombies, Adventure, Survival, Action, Third P...
249050    [Roguelike, Strategy, Tower Defense, Pixel Gra...
Name: tags, dtype: object

####  TF-IDF for measuring semantic similarity between game metadata

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

df_ready['tags_str'] = df_ready['tags'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
df_ready['tags_str'] = df_ready['tags_str'].fillna('')

df_ready.reset_index(inplace=True)
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df_ready['tags_str'])

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

tfidf_matrix.shape


(50872, 467)

In [15]:
# Array mapping from feature integer indices to feature name.
tfidf.get_feature_names_out()

array(['1980s', '1990', '2d', '360', '3d', '40k', '4x', '5d', '6dof',
       'abstract', 'access', 'action', 'addictive', 'adventure',
       'agriculture', 'aliens', 'alternate', 'ambient', 'america',
       'american', 'animation', 'anime', 'apocalyptic', 'arcade',
       'archery', 'arena', 'artificial', 'arts', 'assassin', 'asymmetric',
       'asynchronous', 'atmospheric', 'attack', 'atv', 'audio', 'auto',
       'automation', 'automobile', 'awkward', 'base', 'baseball', 'based',
       'basketball', 'battle', 'battler', 'beat', 'beautiful',
       'benchmark', 'bikes', 'bit', 'blood', 'bmx', 'board', 'book',
       'boss', 'bowling', 'boxing', 'builder', 'building', 'bullet',
       'campaign', 'capitalism', 'card', 'cartoon', 'cartoony', 'casual',
       'cats', 'character', 'chess', 'choices', 'choose', 'cinematic',
       'city', 'class', 'classic', 'click', 'clicker', 'coding', 'cold',
       'collectathon', 'collector', 'colony', 'colorful', 'combat',
       'comedy', 'comic

#### Calculation of similarity coefficients of all games.

In [16]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Compute the cosine similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [17]:
cosine_sim.shape

(50872, 50872)

In [18]:
cosine_sim[1]

array([0.08378862, 1.        , 0.08498079, ..., 0.        , 0.10217484,
       0.        ], shape=(50872,))

In [ ]:
#Construct a reverse map of indices and game titles
indices = pd.Series(df_ready.index, index=df_ready['title']).drop_duplicates()

In [20]:
indices[:10]

title
Prince of Persia: Warrior Within™                       0
BRINK: Agents of Change                                 1
Monaco: What's Yours Is Mine                            2
Escape Dead Island                                      3
Dungeon of the ENDLESS™                                 4
METAL SLUG 3                                            5
Enclave                                                 6
Men of War: Assault Squad 2 - Deluxe Edition upgrade    7
Hyperdimension Neptunia Re;Birth1                       8
The Sum of All Fears                                    9
dtype: int64

In [21]:
# Function that takes in game title as input and outputs most similar games
def get_recommendations(title, cosine_sim=cosine_sim):
    # Get the index of the game that matches the title
    idx = indices[title]

    # Get the pairwsie similarity scores of all games with that game
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the games based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar games
    sim_scores = sim_scores[1:11]

    # Get the game indices
    game_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar games
    return df_ready['title'].iloc[game_indices]


In [22]:
get_recommendations('METAL SLUG 3')

16389                                         METAL SLUG X
18478                                         METAL SLUG 2
7491                                          Super Cyborg
5041                                            METAL SLUG
2650                         Contra Anniversary Collection
6208                                              Platypus
37009                          Retro Classix: Heavy Barrel
43804                                    Jump with Friends
19577    Scott Pilgrim vs. The World™: The Game – Compl...
27261                                        Pier Pressure
Name: title, dtype: str

In [23]:
get_recommendations('Stardew Valley')

1                               BRINK: Agents of Change
2                          Monaco: What's Yours Is Mine
3                                    Escape Dead Island
4                               Dungeon of the ENDLESS™
5                                          METAL SLUG 3
6                                               Enclave
7     Men of War: Assault Squad 2 - Deluxe Edition u...
8                     Hyperdimension Neptunia Re;Birth1
9                                  The Sum of All Fears
10                                           Cold Fear™
Name: title, dtype: str